In [0]:
from pyspark.sql.functions import (
    col,
    lag,
    avg,
    sum,
    stddev,
    round
)
from pyspark.sql.window import Window

df_gold = spark.table(
    "workspace.civicpulse_gold.service_intelligence_daily"
)

print("Gold rows:", df_gold.count())
print("Gold columns:", len(df_gold.columns))

In [0]:
past_30 = (
    Window
    .orderBy("date")
    .rowsBetween(-30, -1)
)

df_ml = (
    df_gold
    .withColumn(
        "past_avg_received_30",
        avg("complaints_received").over(past_30)
    )
)

display(
    df_ml.select(
        "date",
        "complaints_received",
        "past_avg_received_30"
    )
    .orderBy("date")
    .limit(35)
)

In [0]:
from pyspark.sql.functions import (
    col,
    lag,
    avg,
    sum,
    max,
    stddev,
    dayofweek,
    month,
    year
)
from pyspark.sql.window import Window

# Time-based windows using previous observations only
past_7 = Window.orderBy("date").rowsBetween(-7, -1)
past_30 = Window.orderBy("date").rowsBetween(-30, -1)

lag_1 = Window.orderBy("date")

df_ml = (
    df_ml

    # Lag features
    .withColumn(
        "received_lag_1",
        lag("complaints_received", 1).over(lag_1)
    )
    .withColumn(
        "received_lag_7",
        lag("complaints_received", 7).over(lag_1)
    )
    .withColumn(
        "pending_lag_1",
        lag("complaints_pending", 1).over(lag_1)
    )
    .withColumn(
        "pending_lag_7",
        lag("complaints_pending", 7).over(lag_1)
    )

    # Historical demand features
    .withColumn(
        "past_avg_received_7",
        avg("complaints_received").over(past_7)
    )
    .withColumn(
        "past_avg_received_30",
        avg("complaints_received").over(past_30)
    )
    .withColumn(
        "past_max_received_30",
        max("complaints_received").over(past_30)
    )
    .withColumn(
        "past_std_received_30",
        stddev("complaints_received").over(past_30)
    )

    # Historical backlog features
    .withColumn(
        "past_avg_pending_30",
        avg("complaints_pending").over(past_30)
    )
    .withColumn(
        "past_max_pending_30",
        max("complaints_pending").over(past_30)
    )

    # Calendar features
    .withColumn(
        "day_of_week",
        dayofweek("date")
    )
    .withColumn(
        "month",
        month("date")
    )
    .withColumn(
        "year",
        year("date")
    )
)

display(
    df_ml.select(
        "date",
        "complaints_received",
        "complaints_pending",
        "received_lag_1",
        "received_lag_7",
        "pending_lag_1",
        "pending_lag_7",
        "past_avg_received_7",
        "past_avg_received_30",
        "past_max_received_30",
        "past_std_received_30",
        "past_avg_pending_30",
        "past_max_pending_30",
        "day_of_week",
        "month",
        "year"
    )
    .orderBy("date")
    .limit(40)
)

In [0]:
from pyspark.sql.functions import lead
from pyspark.sql.window import Window

date_window = Window.orderBy("date")

df_ml = (
    df_ml
    .withColumn(
        "next_day_received",
        lead("complaints_received", 1).over(date_window)
    )
)

display(
    df_ml.select(
        "date",
        "complaints_received",
        "received_lag_1",
        "received_lag_7",
        "past_avg_received_7",
        "past_avg_received_30",
        "past_max_received_30",
        "past_std_received_30",
        "next_day_received"
    )
    .orderBy("date")
    .limit(40)
)

In [0]:
from pyspark.sql.functions import col

feature_columns = [
    "received_lag_1",
    "received_lag_7",
    "pending_lag_1",
    "pending_lag_7",
    "past_avg_received_7",
    "past_avg_received_30",
    "past_max_received_30",
    "past_std_received_30",
    "past_avg_pending_30",
    "past_max_pending_30",
    "day_of_week",
    "month",
    "year"
]

df_ml_ready = (
    df_ml
    .select(
        "date",
        *feature_columns,
        "next_day_received"
    )
    .dropna()
)

print("Rows before filtering:", df_ml.count())
print("Rows after filtering:", df_ml_ready.count())

display(
    df_ml_ready
    .orderBy("date")
    .limit(10)
)

In [0]:
print("ML rows:", df_ml_ready.count())

display(
    df_ml_ready
    .agg(
        {"date": "min"}
    )
)

display(
    df_ml_ready
    .agg(
        {"date": "max"}
    )
)

In [0]:
from pyspark.sql.functions import min, max, avg, stddev, count

summary = df_ml_ready.agg(
    count("*").alias("rows"),
    min("date").alias("start_date"),
    max("date").alias("end_date"),
    avg("next_day_received").alias("avg_target"),
    stddev("next_day_received").alias("std_target"),
    max("next_day_received").alias("max_target")
)

display(summary)

In [0]:
from pyspark.sql.functions import row_number, count
from pyspark.sql.window import Window

# Add chronological row number
order_window = Window.orderBy("date")

df_split = (
    df_ml_ready
    .withColumn("row_num", row_number().over(order_window))
)

total_rows = df_split.count()

train_end = int(total_rows * 0.70)
validation_end = int(total_rows * 0.85)

df_train = (
    df_split
    .filter(col("row_num") <= train_end)
    .drop("row_num")
)

df_validation = (
    df_split
    .filter(
        (col("row_num") > train_end) &
        (col("row_num") <= validation_end)
    )
    .drop("row_num")
)

df_test = (
    df_split
    .filter(col("row_num") > validation_end)
    .drop("row_num")
)

print("Total:", total_rows)

print(
    "Train:",
    df_train.count(),
    "| Validation:",
    df_validation.count(),
    "| Test:",
    df_test.count()
)

display(
    df_train.agg(
        min("date").alias("start"),
        max("date").alias("end")
    )
)

display(
    df_validation.agg(
        min("date").alias("start"),
        max("date").alias("end")
    )
)

display(
    df_test.agg(
        min("date").alias("start"),
        max("date").alias("end")
    )
)

In [0]:
from pyspark.sql.functions import (
    abs,
    avg,
    sqrt,
    pow
)

df_baseline = (
    df_test
    .withColumn(
        "baseline_prediction",
        col("received_lag_1")
    )
    .withColumn(
        "absolute_error",
        abs(col("next_day_received") - col("baseline_prediction"))
    )
    .withColumn(
        "squared_error",
        pow(
            col("next_day_received") - col("baseline_prediction"),
            2
        )
    )
)

baseline_metrics = df_baseline.agg(
    avg("absolute_error").alias("MAE"),
    sqrt(avg("squared_error")).alias("RMSE")
)

display(baseline_metrics)

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor

feature_columns = [
    "received_lag_1",
    "received_lag_7",
    "pending_lag_1",
    "pending_lag_7",
    "past_avg_received_7",
    "past_avg_received_30",
    "past_max_received_30",
    "past_std_received_30",
    "past_avg_pending_30",
    "past_max_pending_30",
    "day_of_week",
    "month",
    "year"
]

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

train_vector = assembler.transform(df_train)
validation_vector = assembler.transform(df_validation)
test_vector = assembler.transform(df_test)

display(
    train_vector.select(
        "date",
        "features",
        "next_day_received"
    ).limit(10)
)

In [0]:
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="next_day_received",
    predictionCol="prediction",
    numTrees=200,
    maxDepth=10,
    seed=42
)

rf_model = rf.fit(train_vector)

print("Random Forest trained successfully.")

In [0]:
from pyspark.sql.functions import abs, avg, sqrt, pow

# Generate validation predictions
validation_predictions = rf_model.transform(validation_vector)

# Calculate validation metrics
validation_metrics = validation_predictions.agg(
    avg(
        abs(col("next_day_received") - col("prediction"))
    ).alias("MAE"),
    sqrt(
        avg(
            pow(
                col("next_day_received") - col("prediction"),
                2
            )
        )
    ).alias("RMSE")
)

display(validation_metrics)

In [0]:
display(
    validation_predictions
    .select(
        "date",
        "next_day_received",
        "prediction"
    )
    .orderBy("date")
    .limit(20)
)

In [0]:
from pyspark.sql.functions import abs, avg, sqrt, pow

validation_baseline = (
    df_validation
    .withColumn(
        "baseline_prediction",
        col("received_lag_1")
    )
    .withColumn(
        "absolute_error",
        abs(col("next_day_received") - col("baseline_prediction"))
    )
    .withColumn(
        "squared_error",
        pow(
            col("next_day_received") - col("baseline_prediction"),
            2
        )
    )
)

validation_baseline_metrics = validation_baseline.agg(
    avg("absolute_error").alias("MAE"),
    sqrt(avg("squared_error")).alias("RMSE")
)

display(validation_baseline_metrics)

In [0]:
import pandas as pd

importance_data = list(
    zip(
        feature_columns,
        rf_model.featureImportances.toArray()
    )
)

importance_df = pd.DataFrame(
    importance_data,
    columns=["feature", "importance"]
).sort_values(
    "importance",
    ascending=False
)

display(importance_df)

In [0]:
from pyspark.ml.regression import GBTRegressor

gbt = GBTRegressor(
    featuresCol="features",
    labelCol="next_day_received",
    predictionCol="prediction",
    maxIter=100,
    maxDepth=5,
    stepSize=0.05,
    seed=42
)

gbt_model = gbt.fit(train_vector)

print("GBT trained successfully.")

In [0]:
gbt_validation_predictions = gbt_model.transform(validation_vector)

gbt_validation_metrics = gbt_validation_predictions.agg(
    avg(
        abs(col("next_day_received") - col("prediction"))
    ).alias("MAE"),
    sqrt(
        avg(
            pow(
                col("next_day_received") - col("prediction"),
                2
            )
        )
    ).alias("RMSE")
)

display(gbt_validation_metrics)

In [0]:
display(
    gbt_validation_predictions
    .select(
        "date",
        "next_day_received",
        "prediction"
    )
    .orderBy("date")
    .limit(20)
)

In [0]:
from pyspark.sql.functions import (
    avg,
    abs,
    pow,
    sqrt,
    col
)
from pyspark.sql.window import Window

# Get the validation date boundaries
validation_start = df_validation.agg(
    {"date": "min"}
).collect()[0][0]

validation_end = df_validation.agg(
    {"date": "max"}
).collect()[0][0]

# Historical 7-observation window
past_7_baseline = (
    Window
    .orderBy("date")
    .rowsBetween(-7, -1)
)

# Build baseline from the original data that still has complaints_received
df_moving_baseline = (
    df_ml
    .withColumn(
        "moving_avg_prediction",
        avg("complaints_received").over(past_7_baseline)
    )
    .filter(
        (col("date") >= validation_start) &
        (col("date") <= validation_end)
    )
    .withColumn(
        "absolute_error",
        abs(
            col("next_day_received") -
            col("moving_avg_prediction")
        )
    )
    .withColumn(
        "squared_error",
        pow(
            col("next_day_received") -
            col("moving_avg_prediction"),
            2
        )
    )
    .dropna(subset=["moving_avg_prediction"])
)

moving_baseline_metrics = df_moving_baseline.agg(
    avg("absolute_error").alias("MAE"),
    sqrt(avg("squared_error")).alias("RMSE")
)

display(moving_baseline_metrics)

In [0]:
from pyspark.sql.functions import avg, abs, pow, sqrt, col
from pyspark.sql.window import Window

past_7 = (
    Window
    .orderBy("date")
    .rowsBetween(-7, -1)
)

df_test_baseline = (
    df_ml
    .withColumn(
        "prediction",
        avg("complaints_received").over(past_7)
    )
    .filter(
        (col("date") >= "2025-05-07") &
        (col("date") <= "2026-09-23")
    )
    .dropna(subset=["prediction"])
    .withColumn(
        "absolute_error",
        abs(col("next_day_received") - col("prediction"))
    )
    .withColumn(
        "squared_error",
        pow(col("next_day_received") - col("prediction"), 2)
    )
)

test_baseline_metrics = df_test_baseline.agg(
    avg("absolute_error").alias("MAE"),
    sqrt(avg("squared_error")).alias("RMSE")
)

display(test_baseline_metrics)

In [0]:
from pyspark.sql.functions import (
    col,
    when,
    abs
)

df_anomaly = (
    df_ml
    .withColumn(
        "demand_z_score",
        when(
            col("past_std_received_30") > 0,
            (
                col("complaints_received")
                - col("past_avg_received_30")
            ) / col("past_std_received_30")
        )
    )
    .withColumn(
        "demand_anomaly_flag",
        when(
            col("demand_z_score") >= 3,
            1
        ).otherwise(0)
    )
)

display(
    df_anomaly
    .select(
        "date",
        "complaints_received",
        "past_avg_received_30",
        "past_std_received_30",
        "demand_z_score",
        "demand_anomaly_flag"
    )
    .filter(col("demand_anomaly_flag") == 1)
    .orderBy(col("demand_z_score").desc())
    .limit(20)
)

In [0]:
from pyspark.sql.functions import when, col

df_anomaly = (
    df_anomaly
    .withColumn(
        "anomaly_type",
        when(
            (col("demand_z_score") >= 3) &
            (col("complaints_received") >= 314),
            "OPERATIONAL_SURGE"
        )
        .when(
            col("demand_z_score") >= 3,
            "STATISTICAL_ANOMALY"
        )
        .otherwise("NORMAL")
    )
)

display(
    df_anomaly
    .select(
        "date",
        "complaints_received",
        "past_avg_received_30",
        "past_std_received_30",
        "demand_z_score",
        "anomaly_type"
    )
    .filter(col("anomaly_type") != "NORMAL")
    .orderBy(col("date"))
)

In [0]:
from pyspark.sql.functions import sum, when, col
from pyspark.sql.window import Window

past_7 = (
    Window
    .orderBy("date")
    .rowsBetween(-7, -1)
)

df_pressure = (
    df_anomaly
    .withColumn(
        "operational_surge_flag",
        when(
            col("anomaly_type") == "OPERATIONAL_SURGE",
            1
        ).otherwise(0)
    )
    .withColumn(
        "operational_surges_7obs",
        sum("operational_surge_flag").over(past_7)
    )
    .withColumn(
        "surge_persistence",
        when(
            col("operational_surges_7obs") >= 2,
            "SUSTAINED"
        )
        .when(
            col("operational_surges_7obs") == 1,
            "RECENT_SURGE"
        )
        .otherwise("NO_RECENT_SURGE")
    )
)

display(
    df_pressure
    .select(
        "date",
        "complaints_received",
        "demand_z_score",
        "anomaly_type",
        "operational_surges_7obs",
        "surge_persistence"
    )
    .filter(
        col("anomaly_type") == "OPERATIONAL_SURGE"
    )
    .orderBy("date")
)

In [0]:
from pyspark.sql.functions import (
    col,
    when,
    round
)

BACKLOG_P95 = 8950

df_pressure = (
    df_pressure
    .withColumn(
        "backlog_pressure_ratio",
        round(
            col("complaints_pending") / BACKLOG_P95,
            2
        )
    )
    .withColumn(
        "backlog_pressure_flag",
        when(
            col("complaints_pending") >= BACKLOG_P95,
            1
        ).otherwise(0)
    )
)

display(
    df_pressure
    .select(
        "date",
        "complaints_received",
        "complaints_pending",
        "demand_z_score",
        "anomaly_type",
        "operational_surges_7obs",
        "surge_persistence",
        "backlog_pressure_ratio",
        "backlog_pressure_flag"
    )
    .filter(
        (col("anomaly_type") == "OPERATIONAL_SURGE") |
        (col("backlog_pressure_flag") == 1)
    )
    .orderBy("date")
)

In [0]:
from pyspark.sql.functions import col, when

df_intervention = (
    df_pressure
    .withColumn(
        "pressure_state",
        when(
            (col("anomaly_type") == "OPERATIONAL_SURGE") &
            (col("surge_persistence") == "SUSTAINED") &
            (col("backlog_pressure_flag") == 1),
            "CRITICAL_COMBINED_PRESSURE"
        )
        .when(
            (col("anomaly_type") == "OPERATIONAL_SURGE") &
            (col("backlog_pressure_flag") == 1),
            "COMBINED_PRESSURE"
        )
        .when(
            (col("anomaly_type") == "OPERATIONAL_SURGE") &
            (col("surge_persistence").isin(
                "RECENT_SURGE",
                "SUSTAINED"
            )),
            "PERSISTENT_DEMAND_SURGE"
        )
        .when(
            col("anomaly_type") == "OPERATIONAL_SURGE",
            "ISOLATED_DEMAND_SURGE"
        )
        .when(
            col("backlog_pressure_flag") == 1,
            "BACKLOG_PRESSURE"
        )
        .otherwise("NORMAL")
    )
)

display(
    df_intervention
    .select(
        "date",
        "complaints_received",
        "complaints_pending",
        "demand_z_score",
        "surge_persistence",
        "backlog_pressure_ratio",
        "pressure_state"
    )
    .filter(col("pressure_state") != "NORMAL")
    .orderBy("date")
)

In [0]:
from pyspark.sql.functions import col, count, round

state_summary = (
    df_intervention
    .groupBy("pressure_state")
    .agg(
        count("*").alias("days"),
        round(
            count("*") / df_intervention.count() * 100,
            2
        ).alias("percentage")
    )
    .orderBy(col("days").desc())
)

display(state_summary)

In [0]:
from pyspark.sql.functions import (
    col,
    sum,
    when
)
from pyspark.sql.window import Window

past_7 = (
    Window
    .orderBy("date")
    .rowsBetween(-7, -1)
)

df_intervention = (
    df_intervention
    .withColumn(
        "backlog_pressure_7obs",
        sum(
            when(
                col("backlog_pressure_flag") == 1,
                1
            ).otherwise(0)
        ).over(past_7)
    )
    .withColumn(
        "backlog_persistence",
        when(
            col("backlog_pressure_7obs") >= 5,
            "SUSTAINED_BACKLOG"
        )
        .when(
            col("backlog_pressure_7obs") >= 2,
            "RECENT_BACKLOG"
        )
        .otherwise(
            "NO_RECENT_BACKLOG"
        )
    )
)

display(
    df_intervention
    .select(
        "date",
        "complaints_received",
        "complaints_pending",
        "backlog_pressure_ratio",
        "backlog_pressure_7obs",
        "backlog_persistence",
        "pressure_state"
    )
    .filter(
        col("backlog_pressure_flag") == 1
    )
    .orderBy("date")
)

In [0]:
from pyspark.sql.functions import (
    min,
    max,
    avg,
    percentile_approx,
    col
)

display(
    df_intervention
    .select(
        min("demand_z_score").alias("min_demand_z"),
        max("demand_z_score").alias("max_demand_z"),
        percentile_approx(
            "demand_z_score", 0.95
        ).alias("p95_demand_z"),

        min("backlog_pressure_ratio").alias(
            "min_backlog_ratio"
        ),
        max("backlog_pressure_ratio").alias(
            "max_backlog_ratio"
        ),
        percentile_approx(
            "backlog_pressure_ratio", 0.95
        ).alias("p95_backlog_ratio"),

        max("operational_surges_7obs").alias(
            "max_demand_persistence"
        ),
        max("backlog_pressure_7obs").alias(
            "max_backlog_persistence"
        )
    )
)

In [0]:
import pyspark.sql.functions as F

df_scored = (
    df_intervention

    # 1. Demand intensity
    .withColumn(
        "demand_intensity",
        F.least(
            F.greatest(
                F.col("demand_z_score") / F.lit(2.0364),
                F.lit(0.0)
            ),
            F.lit(1.0)
        )
    )

    # 2. Backlog intensity
    .withColumn(
        "backlog_intensity",
        F.least(
            F.greatest(
                F.col("backlog_pressure_ratio"),
                F.lit(0.0)
            ),
            F.lit(1.0)
        )
    )

    # 3. Demand persistence
    .withColumn(
        "demand_persistence_score",
        F.least(
            F.col("operational_surges_7obs") / F.lit(2.0),
            F.lit(1.0)
        )
    )

    # 4. Backlog persistence
    .withColumn(
        "backlog_persistence_score",
        F.least(
            F.col("backlog_pressure_7obs") / F.lit(7.0),
            F.lit(1.0)
        )
    )

    # 5. Combined intervention pressure score
    .withColumn(
        "intervention_pressure_score",
        F.round(
            (
                F.col("demand_intensity")
                + F.col("backlog_intensity")
                + F.col("demand_persistence_score")
                + F.col("backlog_persistence_score")
            ) / F.lit(4.0) * F.lit(100.0),
            2
        )
    )
)

display(
    df_scored
    .select(
        "date",
        "complaints_received",
        "complaints_pending",
        "demand_intensity",
        "backlog_intensity",
        "demand_persistence_score",
        "backlog_persistence_score",
        "intervention_pressure_score",
        "pressure_state"
    )
    .orderBy(
        F.col("intervention_pressure_score").desc()
    )
    .limit(20)
)

In [0]:
display(
    df_scored
    .select(
        F.min("intervention_pressure_score").alias("min_score"),
        F.max("intervention_pressure_score").alias("max_score"),
        F.avg("intervention_pressure_score").alias("avg_score"),
        F.expr("percentile(intervention_pressure_score, 0.25)").alias("p25"),
        F.expr("percentile(intervention_pressure_score, 0.50)").alias("p50"),
        F.expr("percentile(intervention_pressure_score, 0.75)").alias("p75"),
        F.expr("percentile(intervention_pressure_score, 0.90)").alias("p90"),
        F.expr("percentile(intervention_pressure_score, 0.95)").alias("p95")
    )
)

In [0]:
df_scored = (
    df_scored
    .withColumn(
        "intervention_pressure_band",
        F.when(
            F.col("intervention_pressure_score") >= 70,
            "VERY_HIGH"
        )
        .when(
            F.col("intervention_pressure_score") >= 50,
            "HIGH"
        )
        .when(
            F.col("intervention_pressure_score") >= 25,
            "ELEVATED"
        )
        .otherwise("LOW")
    )
)

display(
    df_scored
    .groupBy("intervention_pressure_band")
    .count()
    .orderBy("intervention_pressure_band")
)

In [0]:
display(
    df_scored
    .filter(
        F.col("intervention_pressure_band") == "VERY_HIGH"
    )
    .select(
        "date",
        "complaints_received",
        "complaints_pending",
        "demand_z_score",
        "operational_surges_7obs",
        "backlog_pressure_7obs",
        "demand_intensity",
        "backlog_intensity",
        "demand_persistence_score",
        "backlog_persistence_score",
        "intervention_pressure_score",
        "pressure_state",
        "surge_persistence",
        "backlog_persistence"
    )
    .orderBy(F.col("date"))
)

In [0]:
display(
    df_scored.select(
        "date",
        "complaints_received",
        "complaints_pending",
        "demand_z_score",
        "anomaly_type",
        "operational_surges_7obs",
        "surge_persistence",
        "backlog_pressure_7obs",
        "backlog_persistence",
        "pressure_state",
        "intervention_pressure_score",
        "intervention_pressure_band"
    ).orderBy(F.col("date").desc()).limit(20)
)

In [0]:
# Final Gold-layer intervention intelligence

df_gold_intervention = (
    df_scored
    .select(
        "city_name",
        "date",

        # Core service metrics
        "complaints_received",
        "complaints_disposed",
        "complaints_pending",

        # Historical context
        "previous_pending",
        "pending_change",
        "net_complaint_flow",

        # Reconciliation / data-quality signals
        "expected_pending",
        "reconciliation_gap",
        "reconciliation_status",

        # Demand context
        "received_7d",
        "avg_received_7d",
        "received_30d",
        "avg_received_30d",
        "max_pending_7d",

        # ML / anomaly signals
        "demand_surge_ratio",
        "demand_z_score",
        "anomaly_type",

        # Pressure and persistence
        "operational_surges_7obs",
        "surge_persistence",
        "backlog_pressure_ratio",
        "backlog_pressure_flag",
        "backlog_pressure_7obs",
        "backlog_persistence",

        # Operational classification
        "pressure_state",

        # Intervention scoring
        "demand_intensity",
        "backlog_intensity",
        "demand_persistence_score",
        "backlog_persistence_score",
        "intervention_pressure_score",
        "intervention_pressure_band"
    )
)

(
    df_gold_intervention
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.civicpulse_gold.intervention_intelligence_daily"
    )
)

print("Final Gold table created successfully.")
print(
    "Rows:",
    spark.table(
        "workspace.civicpulse_gold.intervention_intelligence_daily"
    ).count()
)

In [0]:
display(
    df_ml
    .select(
        "date",
        "complaints_received",
        "complaints_pending",
        "received_lag_1",
        "received_lag_7",
        "past_avg_received_7",
        "past_avg_received_30"
    )
    .orderBy(F.col("date").desc())
    .limit(10)
)

In [0]:
latest_forecast = (
    df_ml
    .orderBy(F.col("date").desc())
    .limit(1)
    .select(
        "date",
        "complaints_received",
        "complaints_pending",
        "past_avg_received_7",
        "past_avg_received_30"
    )
)

display(latest_forecast)

In [0]:
import pyspark.sql.functions as F

latest_row = (
    df_ml
    .orderBy(F.col("date").desc())
    .limit(1)
)

df_forecast = (
    latest_row
    .select(
        F.col("city_name"),
        F.col("date").alias("forecast_origin_date"),

        F.col("complaints_received").alias(
            "latest_complaints_received"
        ),

        F.round(
            F.col("past_avg_received_7"), 2
        ).alias(
            "forecast_next_observation"
        ),

        F.round(
            F.col("past_avg_received_30"), 2
        ).alias(
            "avg_received_30_context"
        )
    )
    .withColumn(
        "forecast_vs_latest",
        F.round(
            F.col("forecast_next_observation")
            - F.col("latest_complaints_received"),
            2
        )
    )
    .withColumn(
        "forecast_change_pct",
        F.round(
            F.when(
                F.col("latest_complaints_received") != 0,
                (
                    (
                        F.col("forecast_next_observation")
                        - F.col("latest_complaints_received")
                    )
                    / F.col("latest_complaints_received")
                ) * 100
            ).otherwise(None),
            2
        )
    )
)

display(df_forecast)

In [0]:
forecast_table = "workspace.civicpulse_gold.demand_forecast_latest"

(
    df_forecast
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(forecast_table)
)

print("Forecast Gold table created successfully.")
print(
    "Rows:",
    spark.table(forecast_table).count()
)

In [0]:
latest_intervention = (
    spark.table(
        "workspace.civicpulse_gold.intervention_intelligence_daily"
    )
    .orderBy(F.col("date").desc())
    .limit(1)
)

latest_forecast = spark.table(
    "workspace.civicpulse_gold.demand_forecast_latest"
)

display(
    latest_intervention
    .select(
        "city_name",
        "date",
        "complaints_received",
        "complaints_pending",
        "demand_z_score",
        "anomaly_type",
        "surge_persistence",
        "backlog_persistence",
        "pressure_state",
        "intervention_pressure_score",
        "intervention_pressure_band"
    )
)

In [0]:
display(latest_forecast)

In [0]:
import pyspark.sql.functions as F

latest_intervention = (
    spark.table(
        "workspace.civicpulse_gold.intervention_intelligence_daily"
    )
    .orderBy(F.col("date").desc())
    .limit(1)
)

latest_forecast = spark.table(
    "workspace.civicpulse_gold.demand_forecast_latest"
)

df_decision_support = (
    latest_intervention.alias("i")
    .join(
        latest_forecast.alias("f"),
        on=[
            F.col("i.city_name") == F.col("f.city_name"),
            F.col("i.date") == F.col("f.forecast_origin_date")
        ],
        how="inner"
    )
    .select(
        F.col("i.city_name").alias("city_name"),
        F.col("i.date").alias("as_of_date"),

        # Current operational state
        F.col("i.complaints_received"),
        F.col("i.complaints_pending"),

        # Anomaly intelligence
        F.col("i.demand_z_score"),
        F.col("i.anomaly_type"),
        F.col("i.surge_persistence"),
        F.col("i.backlog_persistence"),

        # Intervention intelligence
        F.col("i.pressure_state"),
        F.col("i.intervention_pressure_score"),
        F.col("i.intervention_pressure_band"),

        # Forward demand signal
        F.col("f.forecast_next_observation"),
        F.col("f.avg_received_30_context"),
        F.col("f.forecast_vs_latest"),
        F.col("f.forecast_change_pct")
    )
    .withColumn(
        "decision_signal",
        F.when(
            F.col("intervention_pressure_band") == "VERY_HIGH",
            "IMMEDIATE_REVIEW"
        )
        .when(
            F.col("intervention_pressure_band") == "HIGH",
            "PRIORITY_REVIEW"
        )
        .when(
            F.col("intervention_pressure_band") == "ELEVATED",
            "MONITOR_CLOSELY"
        )
        .otherwise(
            "ROUTINE_MONITORING"
        )
    )
)

display(df_decision_support)

In [0]:
decision_support_table = (
    "workspace.civicpulse_gold.decision_support_latest"
)

(
    df_decision_support
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(decision_support_table)
)

print("Decision Support Gold table created successfully.")
print(
    "Rows:",
    spark.table(decision_support_table).count()
)